In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Configurare la gestione del rumore con Estimator

<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Ti consigliamo di utilizzare queste versioni o versioni più recenti.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Esistono diversi modi per gestire il rumore, tipicamente utilizzando varie tecniche di mitigazione degli errori e soppressione degli errori per evitare che gli errori si verifichino. Queste tecniche causano solitamente un overhead di pre-elaborazione. Pertanto, è importante trovare un equilibrio tra il perfezionamento dei risultati e la garanzia che il job venga completato in un tempo ragionevole.

Estimator supporta le seguenti tecniche di gestione del rumore. Consulta [Tecniche di mitigazione e soppressione degli errori](error-mitigation-and-suppression-techniques) per una spiegazione di ciascuna. Consulta la sezione [Impostazioni personalizzate degli errori](#advanced-error) per le istruzioni su come abilitare queste tecniche.

- [decoupling dinamico](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [twirling di Pauli](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Livello di resilienza
Il `resilience_level` specifica quanta resilienza costruire contro gli errori. Livelli più alti generano risultati più accurati, a spese di tempi di elaborazione più lunghi. I livelli di resilienza possono essere usati per configurare il compromesso costo/accuratezza quando si applica la gestione del rumore alla query della primitiva. La gestione del rumore riduce gli errori (bias) nei risultati elaborando gli output da una raccolta, o ensemble, di Circuit correlati. Il grado di riduzione degli errori dipende dal metodo applicato. Il livello di resilienza astrae la scelta dettagliata del metodo di gestione del rumore per consentire agli utenti di ragionare sul compromesso costo/accuratezza appropriato per la loro applicazione.

Dato questo, ogni livello corrisponde a un metodo o metodi con un livello crescente di overhead di campionamento quantistico per permetterti di sperimentare diversi compromessi tempo-accuratezza. La tabella seguente mostra quali livelli e i corrispondenti metodi sono disponibili per ciascuna delle primitive.

<span id="resilience-table"></span>

| Livello di resilienza | Descrizione | Tecnica |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | Nessuna mitigazione | Nessuna |
| 1 [Predefinito] | Costi minimi di mitigazione: Mitiga gli errori associati agli errori di lettura | [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) measurement twirling |
| 2 | Costi medi di mitigazione. Tipicamente riduce il bias negli Estimator, ma non è garantito che sia zero-bias. | Livello 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) e gate twirling

> **Info:** I livelli di resilienza sono attualmente in beta, quindi l'overhead di campionamento e la qualità della soluzione varieranno da Circuit a Circuit. Nuove funzionalità, opzioni avanzate e strumenti di gestione verranno rilasciati su base continua. Non è garantito che metodi specifici di gestione del rumore vengano applicati a ogni livello di resilienza.

### Configura Estimator con i livelli di resilienza
Puoi usare i livelli di resilienza per specificare le tecniche di gestione del rumore, oppure puoi impostare tecniche personalizzate individualmente come descritto in [Impostazioni personalizzate degli errori](#advanced-error).

> **Note:** Qualsiasi opzione che specifichi manualmente in aggiunta al livello di resilienza viene applicata in aggiunta all'insieme di base di opzioni definite dal livello di resilienza. Quindi, in linea di principio, potresti impostare il livello di resilienza a 1, ma poi disattivare la mitigazione della misura, anche se non è consigliato.
> 
> Ad esempio, impostare il livello di resilienza a 0 disattiva `zne_mitigation`, ma `estimator.options.resilience.zne_mitigation = True` sovrascrive quel valore.

### Esempio
Il codice seguente abilita ZNE, TREX e gate twirling impostando `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Impostazioni personalizzate di gestione del rumore
Puoi attivare e disattivare singoli metodi di gestione del rumore utilizzando le [opzioni di Estimator](/guides/estimator-options).

> **Note:** Non tutte le opzioni funzionano insieme su tutti i tipi di Circuit. Consulta la [tabella di compatibilità delle funzionalità](estimator-options#options-compatibility-table) per i dettagli.

### Esempio

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>